In [ ]:
from datasets import load_dataset

# Dataset has 50 subreddit splits (e.g. 'gaming', 'todayilearned', etc.)
# Loading without a split gives a DatasetDict keyed by subreddit name
reddit = load_dataset("HuggingFaceGECLM/REDDIT_comments", streaming=True)

## Exploratory Analysis

In [ ]:
# Show available splits (subreddits) and peek at the first record
print("Available splits (subreddits):")
print(list(reddit.keys()))
print()

# Peek at the first record from the first split
first_split = list(reddit.keys())[0]
sample = next(iter(reddit[first_split]))
print(f"Fields (from '{first_split}' split):")
print(list(sample.keys()))
print()
for k, v in sample.items():
    preview = str(v)[:200] + "..." if len(str(v)) > 200 else str(v)
    print(f"  {k}: {preview}")

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt

# Sample randomly across subreddit splits, then shard within each split
# for coverage across the full stream.
# created_utc is a Unix timestamp (seconds since epoch).
RECORDS_PER_SHARD  = 20
SHARDS_PER_SPLIT   = 10
TOTAL_SHARDS       = 100
SEED               = 42

random.seed(SEED)
all_splits = list(reddit.keys())
random.shuffle(all_splits)

timestamps = []
subreddits = []

for split_name in all_splits:
    shard_indices = random.sample(range(TOTAL_SHARDS), SHARDS_PER_SPLIT)
    split_count = 0
    for shard_idx in shard_indices:
        shard_ds = reddit[split_name].shard(num_shards=TOTAL_SHARDS, index=shard_idx)
        count = 0
        for record in shard_ds:
            if count >= RECORDS_PER_SHARD:
                break
            ts = record.get("created_utc")
            if ts:
                timestamps.append(int(ts))
                subreddits.append(split_name)
                count += 1
        split_count += count
    print(f"  {split_name:<35}: {split_count} records  (running total: {len(timestamps)})")

print(f"\nSampled {len(timestamps)} records across {len(all_splits)} subreddits")

# Parse Unix timestamps
dates = pd.to_datetime(timestamps, unit="s", utc=True)
s = pd.Series(dates)
by_year = s.dt.year.value_counts().sort_index()
by_ym   = s.dt.to_period("M").value_counts().sort_index()

# Plot date distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

by_year.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title(f"Comments by Year  (n={len(timestamps):,}, all subreddits)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

by_ym_ts = by_ym.copy()
by_ym_ts.index = by_ym_ts.index.to_timestamp()
by_ym_ts.plot(kind="bar", ax=axes[1], color="darkorange", edgecolor="none", width=1.0)
axes[1].set_title(f"Comments by Year-Month  (n={len(timestamps):,}, all subreddits)")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Count")
n_ticks = len(by_ym_ts)
step = max(1, n_ticks // 12)
axes[1].set_xticks(range(0, n_ticks, step))
axes[1].set_xticklabels(
    [f"{by_ym_ts.index[i].year}-{str(by_ym_ts.index[i].month).zfill(2)}"
     for i in range(0, n_ticks, step)],
    rotation=45, ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
# Date range, distribution stats, and record count per subreddit
print("=== Date Range ===")
print(f"  Earliest : {s.min()}")
print(f"  Latest   : {s.max()}")
print(f"  Span     : {s.max() - s.min()}")
print()
print("=== Distribution by Year ===")
for year, count in by_year.items():
    bar = "\u2588" * (count * 40 // by_year.max())
    print(f"  {year}  {count:>6}  {bar}")
print()
print("=== Records per Subreddit (sampled) ===")
sub_counts = pd.Series(subreddits).value_counts().sort_values(ascending=False)
for sub, count in sub_counts.items():
    bar = "\u2588" * (count * 40 // sub_counts.max())
    print(f"  {sub:<35} {count:>5}  {bar}")